# Student Prediction — Clustering Analysis

Unsupervised learning to discover **student/lead segments** without using the target (withdrawn/enrolled). We use K-Means and hierarchical clustering, evaluate with silhouette score and elbow method, then interpret clusters in terms of risk and engagement.

**Note:** If the kernel fails to start (e.g. "timeout waiting for ports"), try running from a terminal: `jupyter notebook notebooks/clustering_analysis.ipynb` from the project root, or select a different Python interpreter. The notebook samples data for clustering so it runs in under a minute.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.facecolor"] = "white"
PALETTE = {"teal": "#2e86ab", "coral": "#e07a5f", "accent": "#c45c26", "success": "#2d6a4f"}

from src.data_generation import RetentionDataGenerator, LeadScoringDataGenerator, load_config
from src.feature_engineering import RetentionFeatureEngineer, LeadScoringFeatureEngineer

config = load_config()
data_dir = PROJECT_ROOT / "data"
data_dir.mkdir(exist_ok=True)
print("Setup OK.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\drake\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\drake\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\drake\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\drake\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\drake\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\drake\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\drake\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\drake\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed

## 1. Load data and build feature matrices

Retention: mid-semester features (one row per student). Lead: merged GA4+CRM+SIS features. We **sample** to a max of 5,000 rows per dataset for clustering so the notebook runs in under a minute.

In [ ]:
# Retention
ret_path = data_dir / "retention_data.csv"
if ret_path.exists():
    retention_df = pd.read_csv(ret_path)
else:
    retention_df = RetentionDataGenerator(config).generate()
    retention_df.to_csv(ret_path, index=False)

fe_ret = RetentionFeatureEngineer()
mid_feat = fe_ret.create_mid_semester_features(retention_df)
ret_cols = fe_ret.get_feature_columns("mid")
X_ret_full = mid_feat[ret_cols].fillna(mid_feat[ret_cols].median()).fillna(0)
y_ret = mid_feat["withdrawn"]

# Lead
ga4_df = pd.read_csv(data_dir / "ga4_data.csv")
crm_df = pd.read_csv(data_dir / "crm_data.csv")
sis_df = pd.read_csv(data_dir / "sis_data.csv")
fe_lead = LeadScoringFeatureEngineer()
merged_lead = fe_lead.merge_sources(ga4_df, crm_df, sis_df)
lead_feat = fe_lead.create_features(merged_lead)
lead_cols = fe_lead.get_feature_columns()
X_lead_full = lead_feat[lead_cols].fillna(lead_feat[lead_cols].median()).fillna(0)
y_lead = lead_feat["enrolled"]

# Sample for clustering so notebook runs quickly (K-Means on 22k+ rows is slow)
MAX_CLUSTER_N = 5000
np.random.seed(42)
if len(X_ret_full) > MAX_CLUSTER_N:
    idx_ret = np.random.choice(len(X_ret_full), MAX_CLUSTER_N, replace=False)
    X_ret = X_ret_full.iloc[idx_ret].reset_index(drop=True)
    y_ret = y_ret.iloc[idx_ret].reset_index(drop=True)
    mid_feat = mid_feat.iloc[idx_ret].reset_index(drop=True)
else:
    X_ret = X_ret_full
if len(X_lead_full) > MAX_CLUSTER_N:
    idx_lead = np.random.choice(len(X_lead_full), MAX_CLUSTER_N, replace=False)
    X_lead = X_lead_full.iloc[idx_lead].reset_index(drop=True)
    y_lead = y_lead.iloc[idx_lead].reset_index(drop=True)
    lead_feat = lead_feat.iloc[idx_lead].reset_index(drop=True)
else:
    X_lead = X_lead_full

print(f"Retention (for clustering): {X_ret.shape[0]} students, {X_ret.shape[1]} features")
print(f"Lead (for clustering): {X_lead.shape[0]} leads, {X_lead.shape[1]} features")

## 2. K-Means on retention features — elbow and silhouette

Scale features, then run K-Means for k=2..8. Plot within-cluster SSE (elbow) and silhouette score.

In [ ]:
scaler_ret = StandardScaler()
X_ret_scaled = scaler_ret.fit_transform(X_ret)

Ks = range(2, 9)
inertias = []
silhouettes = []
for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_ret_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_ret_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(Ks, inertias, "o-", color=PALETTE["teal"], linewidth=2, markersize=8)
axes[0].set_xlabel("Number of clusters k")
axes[0].set_ylabel("Within-cluster SSE (inertia)")
axes[0].set_title("Elbow Method — Retention Features")
axes[0].grid(True, alpha=0.3)

axes[1].plot(Ks, silhouettes, "o-", color=PALETTE["coral"], linewidth=2, markersize=8)
axes[1].set_xlabel("Number of clusters k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette — Retention Features")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

k_best_ret = int(Ks[np.argmax(silhouettes)])
print(f"Best k by silhouette: {k_best_ret}")

## 3. Fit final K-Means and interpret retention clusters

Assign cluster labels and compare **withdrawal rate** and **mean risk/composite_risk** by cluster. Name clusters by behavior.

In [ ]:
k_final = 4  # or k_best_ret
km_ret = KMeans(n_clusters=k_final, random_state=42, n_init=10)
clusters_ret = km_ret.fit_predict(X_ret_scaled)
mid_feat["cluster"] = clusters_ret

agg = mid_feat.groupby("cluster").agg(
    withdrawn_rate=("withdrawn", "mean"),
    mean_composite_risk=("composite_risk", "mean"),
    mean_mid_gpa=("mid_gpa", "mean"),
    count=("student_id", "count"),
).round(4)
agg["withdrawn_pct"] = (agg["withdrawn_rate"] * 100).round(1)
print("Retention clusters vs outcome (withdrawn) and key features:")
display(agg)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(agg.index.astype(str), agg["withdrawn_pct"], color=PALETTE["coral"], edgecolor="white", linewidth=0.5)
ax.set_xlabel("Cluster")
ax.set_ylabel("% Withdrawn")
ax.set_title("Withdrawal Rate by Cluster — Retention")
sns.despine(trim=True, ax=ax)
plt.tight_layout()
plt.show()

## 4. Silhouette plot for retention (one chosen k)

Show per-sample silhouette by cluster to assess cohesion and separation.

In [ ]:
sil_vals = silhouette_samples(X_ret_scaled, clusters_ret)
fig, ax = plt.subplots(figsize=(10, 6))
y_low = 0
for i in range(k_final):
    ith_sil = sil_vals[clusters_ret == i]
    ith_sil.sort()
    size = ith_sil.shape[0]
    y_high = y_low + size
    ax.fill_betweenx(np.arange(y_low, y_high), 0, ith_sil, alpha=0.7, color=plt.cm.Set1(i / k_final))
    ax.text(-0.05, y_low + size / 2 - 10, str(i), fontsize=12, fontweight="600")
    y_low = y_high
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Cluster")
ax.set_title(f"Silhouette plot — Retention (k={k_final})")
ax.axvline(x=silhouette_score(X_ret_scaled, clusters_ret), color="red", linestyle="--", label="Mean silhouette")
ax.legend()
sns.despine(trim=True, ax=ax)
plt.tight_layout()
plt.show()

## 5. Hierarchical clustering (dendrogram) — retention

Sample 500 students for a readable dendrogram; full linkage on Euclidean distance.

In [ ]:
n_sample = min(500, len(X_ret_scaled))
np.random.seed(42)
idx = np.random.choice(len(X_ret_scaled), n_sample, replace=False)
X_sample = X_ret_scaled[idx]

link = linkage(X_sample, method="ward")
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(link, ax=ax, leaf_rotation=90, leaf_font_size=6, color_threshold=0.7 * link[-2, 2])
ax.set_title("Hierarchical clustering (Ward) — Retention (sample)")
ax.set_xlabel("Sample index")
sns.despine(trim=True, ax=ax)
plt.tight_layout()
plt.show()

## 6. K-Means on lead scoring features

Same pipeline: scale, elbow/silhouette, fit k=4, then compare **enrollment rate** and engagement by cluster.

In [ ]:
scaler_lead = StandardScaler()
X_lead_scaled = scaler_lead.fit_transform(X_lead)

inertias_l, sils_l = [], []
for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(X_lead_scaled)
    inertias_l.append(km.inertia_)
    sils_l.append(silhouette_score(X_lead_scaled, lab))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(Ks, inertias_l, "o-", color=PALETTE["teal"], linewidth=2, markersize=8)
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow — Lead features")
axes[1].plot(Ks, sils_l, "o-", color=PALETTE["coral"], linewidth=2, markersize=8)
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].set_title("Silhouette — Lead features")
plt.tight_layout(); plt.show()

k_lead = 4
km_lead = KMeans(n_clusters=k_lead, random_state=42, n_init=10)
lead_feat["cluster"] = km_lead.fit_predict(X_lead_scaled)
agg_lead = lead_feat.groupby("cluster").agg(
    enrollment_rate=("enrolled", "mean"),
    mean_engagement=("engagement_score", "mean"),
    count=("lead_id", "count"),
).round(4)
agg_lead["enrollment_pct"] = (agg_lead["enrollment_rate"] * 100).round(1)
print("Lead clusters vs enrollment and engagement:")
display(agg_lead)

## 7. Takeaway

- **Retention**: Clusters separate students by academic/engagement/risk profile; high-withdrawal clusters align with high composite_risk and low mid_gpa — useful for targeting interventions.
- **Lead**: Clusters separate by engagement and conversion; high-enrollment clusters have higher engagement_score — useful for messaging and prioritization.
- Clustering is **unsupervised** so we did not use withdrawn/enrolled when forming groups; the strong association with outcome after the fact validates that our features capture meaningful structure.